In [7]:
import subprocess
import re
from pathlib import Path

def merge_filteredVcf_file(sampleID, filteredVcfPaths, dirPath):
    """
    Merges filtered VCF files for a given sample ID and chromosome, and removes the original files.

    Args:
        sampleID (str): The sample ID used to identify the VCF files.
        filteredVcfPaths (list of str): List of paths to the filtered VCF files.
        dirPath (str): Directory path where the merged VCF files will be saved.

    Raises:
        subprocess.CalledProcessError: If an error occurs while executing the shell commands.
    """
    #chrs = list(set([namef.split(sampleID + "_")[1].split(".")[0] for i in filteredVcfPaths namef = Path(VCFPath).name]))
    chrs = list(set([
        Path(VCFPath).name.split(sampleID + "_")[1].split(".")[0]
        for VCFPath in filteredVcfPaths
    ]))
    print(chrs)
    for chr in chrs:
        tmpipaths = [j for j in filteredVcfPaths if re.search(sampleID + "_" + str(chr) + ".vcf", j) != None]
        tmpsql = " ".join(tmpipaths)
        if len(tmpsql) <= 3:
            print(f"No filtered VCF files found for {sampleID} on chromosome {chr}. Creating an empty VCF file.")
            cmdempty = f"touch {dirPath}{sampleID}_{chr}_filter.vcf"
            subprocess.run(cmdempty, shell=True, check=True)
            continue
        cmdmerge = f"cat {tmpsql} > {dirPath}{sampleID}_{chr}_filter.vcf"
        cmdrm = f"rm {tmpsql}"
        try:
            subprocess.run(cmdmerge, shell=True, check=True)
            subprocess.run(cmdrm, shell=True, check=True)
        except subprocess.CalledProcessError as e:
            print(f"An error occurred while executing command: {e.cmd}")
            print(f"Return code: {e.returncode}")
            print(f"Output: {e.output}")


In [8]:
sampleID = "P4_cSCC"
filteredVcfPaths = ['/Users/liqun/Desktop/scMutraceReproducibility/Manuscript/Revision/NG/data/scMutrace/P4_cSCC_py/filterVCF/P4_cSCC_1.vcfaa_filter.vcf', 
                    '/Users/liqun/Desktop/scMutraceReproducibility/Manuscript/Revision/NG/data/scMutrace/P4_cSCC_py/filterVCF/P4_cSCC_11.vcfaa_filter.vcf', 
                    '/Users/liqun/Desktop/scMutraceReproducibility/Manuscript/Revision/NG/data/scMutrace/P4_cSCC_py/filterVCF/P4_cSCC_1.vcfab_filter.vcf', 
                    '/Users/liqun/Desktop/scMutraceReproducibility/Manuscript/Revision/NG/data/scMutrace/P4_cSCC_py/filterVCF/P4_cSCC_10.vcfaa_filter.vcf', 
                    '/Users/liqun/Desktop/scMutraceReproducibility/Manuscript/Revision/NG/data/scMutrace/P4_cSCC_py/filterVCF/P4_cSCC_12.vcfaa_filter.vcf']
dirPath = "/Users/liqun/Desktop/scMutraceReproducibility/Manuscript/Revision/NG/data/scMutrace/P4_cSCC_py/filterVCF/"
merge_filteredVcf_file(sampleID, filteredVcfPaths, dirPath)

['12', '1', '10', '11']


In [1]:
import pandas as pd
from scipy.io import mmread

# 读取稀疏矩阵
matrix = mmread("/Users/liqun/Desktop/scMutraceReproducibility/Manuscript/Revision/NG/data/scMutrace/P4_cSCC_py/cellxsnp.mtx").tocsr()

# 读取细胞信息
cells = pd.read_csv("/Users/liqun/Desktop/scMutraceReproducibility/Manuscript/Revision/NG/data/scMutrace/P4_cSCC_py/cellxsnp.cells.tsv", sep="\t")

# 读取突变信息
variants = pd.read_csv("/Users/liqun/Desktop/scMutraceReproducibility/Manuscript/Revision/NG/data/scMutrace/P4_cSCC_py/cellxsnp.variants.tsv", sep="\t")

print("Matrix shape:", matrix.shape)
print("Cells:", len(cells))
print("Variants:", len(variants))
print("Nonzero entries:", matrix.nnz)

Matrix shape: (725, 65)
Cells: 725
Variants: 65
Nonzero entries: 797


/var/folders/yx/vkjdxytj10n2fwzdpmn014bc0000gp/T/ipykernel_5152/2586267195.py:5: DeprecationWarning: The default value for `spmatrix` is changing to `False` in v1.20.
             That means the default return type will be a sparse array.
             Unless you use * instead of @, ** for matrix power, or you depend
             on 2D shapes from e.g. `A.sum(axis=0)` it may not matter to you.
             See the spmatrix to sparray migration guide for details.
             https://docs.scipy.org/doc/scipy/reference/sparse.migration_to_sparray.html
             
  matrix = mmread("/Users/liqun/Desktop/scMutraceReproducibility/Manuscript/Revision/NG/data/scMutrace/P4_cSCC_py/cellxsnp.mtx").tocsr()


In [6]:
import anndata as ad

adata = ad.AnnData(
    X=matrix,
    obs=cells.set_index("cell_id"),
    var=variants.set_index("variant_id")
)

adata.var_names_make_unique()

print(adata)

adata.write_h5ad("/Users/liqun/Desktop/scMutraceReproducibility/Manuscript/Revision/NG/data/scMutrace/P4_cSCC_py/cellxsnp.h5ad")

AnnData object with n_obs × n_vars = 725 × 65
    obs: 'celltype'
    var: 'chr', 'pos', 'ref', 'alt', 'mode', 'SNP', 'INDEL', 'otherFilter'
    layers: None (.X)


In [7]:
import pandas as pd

# 稀疏矩阵转 DataFrame
df = pd.DataFrame.sparse.from_spmatrix(
    adata.X,
    index=adata.obs_names,
    columns=adata.var_names
)

# 显示前 5 行、前 5 列
print(df.iloc[:5, :5])

# 保存为 CSV
df.to_csv("/Users/liqun/Desktop/scMutraceReproducibility/Manuscript/Revision/NG/data/scMutrace/P4_cSCC_py/cellxsnp.csv", index_label="cell_id")

variant_id        chr1:178380879:G>A  chr1:178380879:G>C  chr1:182353029:G>A  \
cell_id                                                                        
AAACCTGAGCCCAACC                   0                   0                   0   
AAAGATGTCCTTGACC                   0                   0                   0   
AAAGCAACAACGATGG                   0                   0                   0   
AACACGTAGTGCCAGA                   0                   0                   0   
AACACGTGTTGTGGCC                   0                   0                   0   

variant_id        chr1:192685578:G>A  chr1:224363010:T>C  
cell_id                                                   
AAACCTGAGCCCAACC                   0                   0  
AAAGATGTCCTTGACC                   0                   0  
AAAGCAACAACGATGG                   0                   0  
AACACGTAGTGCCAGA                   0                   0  
AACACGTGTTGTGGCC                   0                   0  
